<a href="https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: CTR vs. position — CONFIRMED
CTR decreases monotonically as position worsens: 0.48% (1-3) → 0.35% (4-10) → 0.28% (11-20) → 0.13% (20+). This supports the assumption behind FlyRank's CTR-fix logic — better rank does drive better CTR in this slice. One honest caveat worth naming: these absolute CTR values are far lower than typical GSC benchmarks (usually several percent even at poor positions) — the direction is confirmed, but the magnitude looks unusually low for this dataset, worth flagging rather than hiding.

Signal 2: Impressions vs. engagement — CONFIRMED (but weak in magnitude)
avg_scroll_events rises monotonically with impression volume: 0.023 (1-10) → 0.083 (11-100) → 0.295 (100+). This supports treating visibility/volume as a legitimate signal behind quick-win prioritization. Caveat: the absolute values are small — most content-days log few or zero scroll events regardless of bucket, so while the direction holds, the effect size is modest, not dramatic.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Signal 1 (flag-linked): CTR vs position
sig1 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END AS position_bucket,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr,
        COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY position_bucket
    ORDER BY position_bucket
""")
print("Signal 1: CTR vs position")
print(sig1)

# Signal 2: impressions volume vs engagement
sig2 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions = 0 THEN '0'
            WHEN gsc_impressions <= 10 THEN '1-10'
            WHEN gsc_impressions <= 100 THEN '11-100'
            ELSE '100+'
        END AS impressions_bucket,
        AVG(scroll_events) AS avg_scroll_events,
        COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY impressions_bucket
    ORDER BY impressions_bucket
""")
print("Signal 2: Impressions vs engagement")
print(sig2)

Signal 1: CTR vs position


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬──────────────────────┬─────────┐
│ position_bucket │       avg_ctr        │    n    │
│     varchar     │        double        │  int64  │
├─────────────────┼──────────────────────┼─────────┤
│ 1-3             │ 0.004755515699920509 │  727362 │
│ 11-20           │ 0.002769909681133761 │  519223 │
│ 20+             │ 0.001289153117819274 │  908354 │
│ 4-10            │ 0.003472635519611634 │ 1456122 │
└─────────────────┴──────────────────────┴─────────┘

Signal 2: Impressions vs engagement


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬──────────────────────┬─────────┐
│ impressions_bucket │  avg_scroll_events   │    n    │
│      varchar       │        double        │  int64  │
├────────────────────┼──────────────────────┼─────────┤
│ 1-10               │ 0.023343036884396907 │ 1531634 │
│ 100+               │   0.2946585242273127 │  633483 │
│ 11-100             │   0.0826039350104394 │ 1445944 │
└────────────────────┴──────────────────────┴─────────┘



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rule applied to March 2026 (gsc_data_available IS TRUE): 3,611,061 rows scored. 71,386 rows (~2% of the full slice) flagged REVIEW_FOR_RANKING_FIX, using the 95th-percentile cutoff among rows with a positive score.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_avg_position, scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

import numpy as np

def reason_code(row):
    if row['gsc_impressions'] == 0:
        return 'NO_VISIBILITY'
    elif row['gsc_avg_position'] <= 10:
        return 'RANKS_WELL'
    else:
        return 'VISIBLE_BUT_POOR_RANK'

df['reason_code'] = df.apply(reason_code, axis=1)

df['score'] = np.where(
    df['reason_code'] == 'VISIBLE_BUT_POOR_RANK',
    df['gsc_impressions'] * (df['gsc_avg_position'] - 10),
    0
)

threshold = df.loc[df['score'] > 0, 'score'].quantile(0.95)
df['action'] = np.where(df['score'] >= threshold, 'REVIEW_FOR_RANKING_FIX', 'MONITOR')

df_ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Rows written:", len(df_ranked))
print("REVIEW_FOR_RANKING_FIX count:", (df_ranked['action']=='REVIEW_FOR_RANKING_FIX').sum())
print(df_ranked.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows written: 3611061
REVIEW_FOR_RANKING_FIX count: 71386
             client_hash_id           content_hash_id report_date  \
0   client_23a62021009f63c4  content_6530fa9d297c46eb  2026-03-31   
1   client_23a62021009f63c4  content_e6df0936699f5b8f  2026-03-31   
2   client_23a62021009f63c4  content_36e53e9c707674fc  2026-03-09   
3   client_23a62021009f63c4  content_36e53e9c707674fc  2026-03-08   
4   client_23a62021009f63c4  content_36e53e9c707674fc  2026-03-11   
5   client_23a62021009f63c4  content_73aa61dcedebbf30  2026-03-31   
6   client_23a62021009f63c4  content_36e53e9c707674fc  2026-03-10   
7   client_23a62021009f63c4  content_36e53e9c707674fc  2026-03-12   
8   client_23a62021009f63c4  content_73aa61dcedebbf30  2026-03-30   
9   client_23a62021009f63c4  content_c3be4b44668a443e  2026-03-31   
10  client_23a62021009f63c4  content_36e53e9c707674fc  2026-03-07   
11  client_23a62021009f63c4  content_559cdd76da9306de  2026-03-08   
12  client_23a62021009f63c4  content_36e53e9c

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review

The top 20 rows aren't 20 distinct opportunities — they collapse into about 6 unique content items, each appearing on multiple different days, all from a single client (client_23a62021009f63c4). Reviewing by content item rather than by row:

Content item ...6530fa9d (1 day: 03-31)
Action: REVIEW_FOR_RANKING_FIX. Reason code: VISIBLE_BUT_POOR_RANK (5,364 impressions, position 89.8). Confidence: high volume, very poor rank — a strong candidate. What would make it wrong: if position ~90 reflects a genuinely irrelevant query match rather than a fixable on-page/technical issue.

Content item ...e6df0936 (1 day: 03-31)
Action: REVIEW_FOR_RANKING_FIX. Reason code: VISIBLE_BUT_POOR_RANK (14,682 impressions, position 25.0). Confidence: highest impression volume in the top 20 — worth prioritizing. What would make it wrong: if this SERP is too competitive for a ranking fix to realistically help.

Content item ...36e53e9c (10 days: 03-02 through 03-12 — ranks 3,4,6,10,12,13,15,16,18 in the raw queue)
Action: REVIEW_FOR_RANKING_FIX. Reason code: VISIBLE_BUT_POOR_RANK (~7,000-9,400 impressions, position 31-34, consistently). Confidence: high — the same pattern repeats for 10 straight days, suggesting a stable, real ranking problem rather than noise. What would make it wrong: this is really one recurring issue, not 10 separate ones — reviewing it once, not 10 times, is the correct action.

Content item ...73aa61dc (3 days: 03-26, 03-30, 03-31 — ranks 5,8,17)
Action: REVIEW_FOR_RANKING_FIX. Reason code: VISIBLE_BUT_POOR_RANK (~4,000-5,500 impressions, position 47.8-51.4). Confidence: moderate — consistent across 3 days. What would make it wrong: same underlying page appearing 3 times shouldn't be treated as 3 opportunities.

Content item ...c3be4b44 (1 day: 03-31)
Action: REVIEW_FOR_RANKING_FIX. Reason code: VISIBLE_BUT_POOR_RANK (4,781 impressions, position 49.6). Confidence: moderate. What would make it wrong: if this is a seasonal or temporary dip rather than a stable pattern worth fixing.

Content item ...559cdd76 (2 days: 03-08, 03-09 — ranks 11,14)
Action: REVIEW_FOR_RANKING_FIX. Reason code: VISIBLE_BUT_POOR_RANK (~6,000-6,500 impressions, position 36.1-39.6). Confidence: moderate. What would make it wrong: same page counted twice inflates its apparent priority.

Content item ...c367b0ca (1 day: 03-10)
Action: REVIEW_FOR_RANKING_FIX. Reason code: VISIBLE_BUT_POOR_RANK (4,166 impressions, position 48.7). Confidence: lower — single-day signal only. What would make it wrong: if this is a one-off dip rather than a persistent ranking problem.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick pattern (the real finding): The top-20 is not a diverse opportunity list — it's ~6 unique content items surfacing repeatedly across different days, entirely from one client. A content team reading this queue "as-is" would think they have 20 different problems to fix, when they actually have about 6. What this rule needs to be genuinely useful: aggregate to one row per content_hash_id (e.g. average or max score across the month) before ranking, rather than ranking raw daily rows. This is the single most important fix before this baseline is trustworthy for a real content team.

Leakage check: The score uses only gsc_impressions and gsc_avg_position, both logged on the same report_date as the row being scored — no future-dated columns, no other month's data, and no client/content identifiers used as if they were signals. Confirmed clean: no future-window or label-derived inputs.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.